# Explain/Eval Completion Report

This notebook checks explain/eval completion for a list of `run_id` values.

Default behavior:
- prints `all complete` if every requested run is complete
- otherwise prints only incomplete or failed runs

Set `DETAILED_REPORT = True` in the configuration cell if you want a full per-run report.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = None

## Configure Input

In [2]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "federated").exists():
    candidate = REPO_ROOT.parent
    if (candidate / "federated").exists():
        REPO_ROOT = candidate

SUMMARY_PATH = REPO_ROOT / "job_launcher/plans/job_launcher__launcher__20260506T145108/post_training_plans_summary.json"

# If RUN_IDS is empty and SUMMARY_PATH exists, the notebook will use every run_id from the summary.
RUN_IDS = []

DETAILED_REPORT = False
SHOW_MISSING_PATHS_PER_RUN = 5

## Helpers

In [3]:
def load_summary(summary_path: Path) -> dict:
    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")

    text = summary_path.read_text(encoding="utf-8")
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = None

    if isinstance(payload, dict):
        runs = payload.get("runs")
        if isinstance(runs, list):
            return payload

    # Allow a JSONL file that already contains one run record per line.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        jsonl_rows = [json.loads(line) for line in lines]
        if all(isinstance(row, dict) and "run_id" in row and "plan_summary" in row for row in jsonl_rows):
            return {"runs": jsonl_rows}
        if all(isinstance(row, dict) and "experiment_index" in row for row in jsonl_rows):
            sibling_summary = summary_path.with_name("post_training_plans_summary.json")
            detail = (
                f"The provided file looks like the pre-training launcher experiment list: {summary_path}. "
                "Use the post-training explain/eval summary instead"
            )
            if sibling_summary.exists():
                detail += f", for example: {sibling_summary}"
            raise ValueError(detail)

    raise ValueError(
        "SUMMARY_PATH must point to a post-training explain/eval summary with a 'runs' list, "
        "or a JSONL file containing run records with 'run_id' and 'plan_summary'. "
        f"Got: {summary_path}"
    )


def resolve_run_ids(summary_payload: dict, configured_run_ids: list[str]) -> list[str]:
    if configured_run_ids:
        return configured_run_ids
    return [run["run_id"] for run in summary_payload.get("runs", [])]


def build_expected_map(summary_payload: dict) -> dict[str, dict]:
    expected = {}
    for run in summary_payload.get("runs", []):
        run_id = run["run_id"]
        expected[run_id] = {
            "job_count": int((run.get("plan_summary") or {}).get("job_count", 0)),
            "plan_path": (run.get("plan_summary") or {}).get("plan_path"),
            "array_range": (run.get("plan_summary") or {}).get("array_range"),
        }
    return expected


def iter_plan_rows(plan_path: Path):
    with plan_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def expected_done_path(repo_root: Path, row: dict) -> Path:
    return (
        repo_root
        / "federated/runs"
        / row["run_id"]
        / "clients"
        / row["client_id"]
        / "selections"
        / row["selection_id"]
        / "shards"
        / row["shard_id"]
        / "_status"
        / f"{row['config_id']}.done"
    )


def expected_status_json_path(repo_root: Path, row: dict) -> Path:
    return expected_done_path(repo_root, row).with_suffix(".json")


def inspect_run(repo_root: Path, run_id: str, expected_info: dict | None) -> dict:
    run_dir = repo_root / "federated/runs" / run_id
    record = {
        "run_id": run_id,
        "run_dir_exists": run_dir.exists(),
        "expected_jobs": None,
        "actual_done": 0,
        "actual_status_json": 0,
        "missing_done": None,
        "missing_paths": [],
        "missing_by_explainer": {},
        "status": "unknown",
        "notes": [],
    }

    if expected_info is None:
        record["status"] = "missing_from_summary"
        record["notes"].append("run_id not present in summary")
        if run_dir.exists():
            record["actual_done"] = sum(1 for _ in run_dir.glob("clients/*/selections/*/shards/*/_status/*.done"))
            record["actual_status_json"] = sum(1 for _ in run_dir.glob("clients/*/selections/*/shards/*/_status/*.json"))
        return record

    record["expected_jobs"] = int(expected_info["job_count"])
    plan_path = repo_root / expected_info["plan_path"]
    if not plan_path.exists():
        record["status"] = "missing_plan"
        record["notes"].append(f"plan file not found: {plan_path}")
        return record

    missing_counter = Counter()
    actual_done = 0
    actual_status_json = 0
    missing_paths = []

    for row in iter_plan_rows(plan_path):
        done_path = expected_done_path(repo_root, row)
        status_json_path = expected_status_json_path(repo_root, row)
        if done_path.exists():
            actual_done += 1
        else:
            missing_counter[row["explainer"]] += 1
            if len(missing_paths) < SHOW_MISSING_PATHS_PER_RUN:
                missing_paths.append(str(done_path.relative_to(repo_root)))
        if status_json_path.exists():
            actual_status_json += 1

    record["actual_done"] = actual_done
    record["actual_status_json"] = actual_status_json
    record["missing_done"] = record["expected_jobs"] - actual_done
    record["missing_paths"] = missing_paths
    record["missing_by_explainer"] = dict(sorted(missing_counter.items()))

    if not run_dir.exists():
        record["status"] = "missing_run_dir"
        record["notes"].append(f"run directory not found: {run_dir}")
    elif record["missing_done"] == 0:
        record["status"] = "complete"
    else:
        record["status"] = "incomplete"
        record["notes"].append(
            f"missing {record['missing_done']} done markers out of {record['expected_jobs']} expected jobs"
        )

    return record


def compact_table(records: list[dict]) -> list[dict]:
    return [
        {
            "run_id": record["run_id"],
            "status": record["status"],
            "expected_jobs": record["expected_jobs"],
            "actual_done": record["actual_done"],
            "missing_done": record["missing_done"],
            "actual_status_json": record["actual_status_json"],
        }
        for record in records
    ]


def maybe_display_table(rows: list[dict]):
    if not rows:
        return
    try:
        import pandas as pd
    except ModuleNotFoundError:
        for row in rows:
            print(row)
        return
    df = pd.DataFrame(rows)
    if display is not None:
        display(df)
    else:
        print(df.to_string(index=False))

## Run Report

In [4]:
summary_payload = load_summary(SUMMARY_PATH)
expected_map = build_expected_map(summary_payload)
run_ids = resolve_run_ids(summary_payload, RUN_IDS)
if not run_ids:
    raise ValueError(
        "No run_ids found to inspect. Check SUMMARY_PATH or set RUN_IDS explicitly."
    )

records = [inspect_run(REPO_ROOT, run_id, expected_map.get(run_id)) for run_id in run_ids]
failed_records = [record for record in records if record["status"] != "complete"]

report = {
    "requested_run_count": len(run_ids),
    "complete_run_count": sum(1 for record in records if record["status"] == "complete"),
    "failed_run_count": len(failed_records),
    "all_complete": len(failed_records) == 0,
}

if report["all_complete"]:
    print("all complete")
else:
    print(json.dumps(report, indent=2))
    maybe_display_table(compact_table(failed_records))

    for record in failed_records:
        print(f"\nrun_id: {record['run_id']}")
        for note in record["notes"]:
            print(f"  note: {note}")
        if record["missing_by_explainer"]:
            print(f"  missing_by_explainer: {record['missing_by_explainer']}")
        if record["missing_paths"]:
            print("  sample_missing_done_paths:")
            for path in record["missing_paths"]:
                print(f"    - {path}")

if DETAILED_REPORT:
    print("\nDetailed per-run report")
    maybe_display_table(compact_table(records))

all complete
